# Практическая работа № 2
## Верификация и оценка качества образовательных данных: от raw-слоя к clean-слою в Google Colab и BigQuery

**Курс:** «Методы анализа больших данных» · **Раздел 2:** «Получение и подготовка больших данных» · **Лекция 2**

| Поле | Заполните |
|---|---|
| ФИО |  |
| Группа |  |
| Номер варианта |  |
| Дата |  |

### Сквозной кейс

Вы получили выгрузку результатов заданий из LMS. Прежде чем считать средние баллы и строить модели,
нужно ответить на вопрос лекции 2: **можно ли этим данным доверять?**

**raw CSV → профилирование → правила качества → конвейер очистки → clean-слой в pandas и BigQuery → сверка → паспорт данных**

### Что важно

1. Все работают с одним файлом **`practice02_raw.csv`**, вариант задаёт **курс и поток** (срез данных).
2. В выгрузке есть типичные дефекты. **Какие именно — вы находите сами** на шаге профилирования.
3. Строки с проблемными значениями **не удаляются молча**: значение заменяется пропуском, причина записывается во флаг.
4. Ячейки с пометкой **`TODO`** нужно дополнить. Автопроверки после них подскажут, если что-то не так.
5. Выполняйте ячейки сверху вниз. После правки `TODO` перезапустите ячейки ниже.

## Теоретический минимум

### Слои данных (лекция 2, раздел 1)

| Слой | Где у нас | Правило |
|---|---|---|
| **raw** | `practice02_raw.csv`, таблица `p02_raw` | хранится «как есть», все поля — текст, не правится вручную |
| **clean** | `clean` в pandas, представление `p02_clean` | каждое изменение — код; причина фиксируется флагом |
| **analytic** | `p02_analytic` (по желанию) | агрегаты под конкретную задачу |

### Поля файла

| Поле | Смысл |
|---|---|
| `event_id` | идентификатор записи в выгрузке |
| `code_module`, `code_presentation` | курс и поток (ваш вариант) |
| `id_assessment`, `assessment_type` | задание и его тип: `TMA` — работа с проверкой тьютором, `CMA` — компьютерный тест, `Exam` |
| `assessment_deadline`, `weight` | срок сдачи (день от начала курса) и вес задания |
| `id_student`, `date_submitted` | обезличенный студент и день сдачи |
| `is_banked` | 1 — результат перезачтён из прошлого потока |
| `score` | балл по шкале 0–100 |
| `final_result` | итог курса: Pass, Distinction, Fail, Withdrawn |

### Показатели качества (лекция 2, раздел 10)

| Показатель | Как считаем в этой работе |
|---|---|
| **C** — полнота | доля строк, где `score` известен (не пусто и не служебный код) |
| **V** — валидность | доля значений в шкале 0–100 **среди заполненных** |
| **U** — уникальность | число уникальных ключей `(id_assessment, id_student)` / число строк |
| **S** — согласованность | доля строк, где тип задания записан по словарю и нет противоречия «Withdrawn + Exam» |

$$DQ = 0{,}35C + 0{,}30V + 0{,}20U + 0{,}15S$$

Веса — учебная договорённость, а не стандарт. Высокое DQ может скрывать проблему в одном измерении.

### Два вида дублей (лекция 2, раздел 8)

- **точный дубль** — строка повторяется целиком;
- **логический дубль** — та же сдача (то же задание, тот же студент) пришла повторно с другим `event_id`.

## Шаг 0. Ваш вариант

| № | Модуль | Поток | № | Модуль | Поток |
|---:|---|---|---:|---|---|
| 1 | `AAA` | `2013J` | 11 | `EEE` | `2013J` |
| 2 | `BBB` | `2013B` | 12 | `EEE` | `2014B` |
| 3 | `BBB` | `2014B` | 13 | `EEE` | `2014J` |
| 4 | `BBB` | `2014J` | 14 | `FFF` | `2013B` |
| 5 | `CCC` | `2014B` | 15 | `FFF` | `2013J` |
| 6 | `CCC` | `2014J` | 16 | `FFF` | `2014B` |
| 7 | `DDD` | `2013B` | 17 | `FFF` | `2014J` |
| 8 | `DDD` | `2013J` | 18 | `GGG` | `2013J` |
| 9 | `DDD` | `2014B` | 19 | `GGG` | `2014B` |
| 10 | `DDD` | `2014J` | 20 | `GGG` | `2014J` |

Укажите номер варианта — курс и поток подставятся автоматически.

In [ ]:
# TODO-0. Номер вашего варианта (1–20)
VARIANT = None

VARIANTS = {
    1: ("AAA", "2013J"),
    2: ("BBB", "2013B"),
    3: ("BBB", "2014B"),
    4: ("BBB", "2014J"),
    5: ("CCC", "2014B"),
    6: ("CCC", "2014J"),
    7: ("DDD", "2013B"),
    8: ("DDD", "2013J"),
    9: ("DDD", "2014B"),
    10: ("DDD", "2014J"),
    11: ("EEE", "2013J"),
    12: ("EEE", "2014B"),
    13: ("EEE", "2014J"),
    14: ("FFF", "2013B"),
    15: ("FFF", "2013J"),
    16: ("FFF", "2014B"),
    17: ("FFF", "2014J"),
    18: ("GGG", "2013J"),
    19: ("GGG", "2014B"),
    20: ("GGG", "2014J"),
}

assert VARIANT in VARIANTS, "Укажите номер варианта от 1 до 20 в переменной VARIANT"
MODULE, PRESENTATION = VARIANTS[VARIANT]
print(f"Вариант {VARIANT}: курс {MODULE}, поток {PRESENTATION}")

## Шаг 1. Библиотеки и загрузка raw-файла

Выполните ячейки. Colab предложит выбрать файл — выберите **`practice02_raw.csv`**, выданный преподавателем.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

In [ ]:
DATA_PATH = Path("practice02_raw.csv")
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Файл не выбран.")
    DATA_PATH = Path(next(iter(uploaded)))
print("Файл:", DATA_PATH)

In [ ]:
# Весь файл читаем как ТЕКСТ (dtype=str): raw-слой хранится «как есть»,
# служебные коды вроде '?' не должны молча превратиться в NaN или число.
raw_all = pd.read_csv(DATA_PATH, dtype=str)

n_pres = raw_all[["code_module", "code_presentation"]].drop_duplicates().shape[0]
print(f"Весь файл: строк {len(raw_all):,}, столбцов {raw_all.shape[1]}, "
      f"уникальных event_id {raw_all['event_id'].nunique():,}, презентаций курсов {n_pres}")

raw = raw_all[(raw_all["code_module"] == MODULE)
              & (raw_all["code_presentation"] == PRESENTATION)].copy()
print(f"Срез {MODULE} {PRESENTATION}: строк {len(raw):,}")
assert len(raw) > 0, "Срез пуст: проверьте MODULE и PRESENTATION"
raw.head()

### Контрольные значения

Для всего файла у всех студентов должно получиться:

```text
строк:               176172
столбцов:            12
уникальных event_id: 174781
презентаций курсов:  22
```

Если значения другие — **не продолжайте**, проверьте файл. Число строк вашего среза сверьте с таблицей вариантов в `README`.

> Почему `dtype=str`? По умолчанию pandas превратит `'999'` и `'-1'` в обычные числа, и они молча войдут в средний балл.
> Raw-слой читается «как есть», а типы приводятся позже — осознанно.

## Шаг 2. Профилирование raw-слоя: что не так с данными?

Готовая ячейка показывает шесть «рентгеновских снимков» вашего среза. Внимательно прочитайте вывод.

In [ ]:
KEY_CANDIDATE = ["id_assessment", "id_student"]   # «кто сдал какое задание»

print("1) Типы столбцов raw-слоя:", raw.dtypes.astype(str).unique().tolist())
print("   Пустых ячеек (NaN):", int(raw.isna().sum().sum()))

num = pd.to_numeric(raw["score"], errors="coerce")
print("\n2) Нечисловые значения score:")
print(raw.loc[num.isna(), "score"].value_counts(dropna=False).to_string())

print("\n3) Числовые значения score вне шкалы 0–100 (значение → сколько раз):")
print(raw.loc[num.notna() & ~num.between(0, 100), "score"].value_counts().to_string())

print("\n4) Написание assessment_type:")
print(raw["assessment_type"].value_counts().to_string())

print("\n5) assessment_deadline = '?':", int((raw["assessment_deadline"] == "?").sum()),
      "| из них по типам:", raw.loc[raw["assessment_deadline"] == "?", "assessment_type"]
      .str.upper().value_counts().to_dict())

n_exact = int(raw.duplicated().sum())
n_event_rep = int(raw["event_id"].duplicated().sum())
n_key_rep = int(raw.drop_duplicates().duplicated(subset=KEY_CANDIDATE).sum())
print("\n6) Полностью одинаковых строк (точные дубли):", n_exact)
print("   Повторов event_id:", n_event_rep)
print("   Повторов пары (id_assessment, id_student) после удаления точных дублей:", n_key_rep)

### 2б. Значения `score` вне шкалы во всём файле

Чтобы отличить служебный код от ошибки ввода, посмотрим на частоты по всем 22 потокам.

In [ ]:
num_all = pd.to_numeric(raw_all["score"], errors="coerce")
out_all = raw_all.loc[num_all.notna() & ~num_all.between(0, 100), "score"].value_counts()
print("Значения вне шкалы 0–100 во всём файле (значение → сколько раз):")
print(out_all.to_string())

### Ваши выводы по профилированию

Ответьте кратко (по 1–2 предложения):

1. Сколько пустых ячеек (`NaN`) в срезе? Значит ли это, что пропусков нет?
2. Какие значения `score` означают **«нет данных»** (служебные коды)? Почему вы так решили?
   *Подсказка:* одно и то же невозможное значение, которое повторяется сотни раз, — это код;
   разные значения чуть выше шкалы, по несколько раз каждое, — ошибки ввода.
3. Какие варианты написания `assessment_type` встречаются?
4. Сколько точных и сколько логических дублей? Чем они отличаются?

> **Ваши выводы:**
> 1. ...
> 2. ...
> 3. ...
> 4. ...

## Шаг 3. Правила качества (TODO)

Перенесите выводы профилирования в правила — это и есть **контракт данных** (лекция 2, раздел 6).

In [ ]:
# TODO-1. Служебные коды «нет данных» в поле score (строки!). Один код уже указан — дополните множество.
MISSING_CODES = {"?"}

# Ключ уникальности события «студент сдал задание»
KEY = ["id_assessment", "id_student"]

# TODO-2. Словарь нормализации типа задания.
# Ключ — значение после .strip().upper(), значение — эталонное написание из словаря {TMA, CMA, Exam}.
TYPE_MAP = {"TMA": "TMA"}

In [ ]:
# Автопроверка правил (по всему файлу)
msgs = []
for value, cnt in out_all.items():
    if cnt >= 50 and value not in MISSING_CODES:
        msgs.append(f"значение {value!r} встречается {cnt} раз — похоже на служебный код; добавьте его в MISSING_CODES")
    if cnt < 50 and value in MISSING_CODES:
        msgs.append(f"значение {value!r} встречается всего {cnt} раз — это ошибка ввода, а не код; уберите из MISSING_CODES")
non_numeric = set(raw_all.loc[num_all.isna(), "score"].dropna())
if not non_numeric <= MISSING_CODES:
    msgs.append(f"нечисловые значения {non_numeric - MISSING_CODES} не учтены в MISSING_CODES")
spellings = set(raw_all["assessment_type"].str.strip().str.upper())
if not spellings <= set(TYPE_MAP):
    msgs.append(f"в TYPE_MAP нет ключей для {spellings - set(TYPE_MAP)}")
if not set(TYPE_MAP.values()) <= {"TMA", "CMA", "Exam"}:
    msgs.append("значения TYPE_MAP должны быть из словаря {TMA, CMA, Exam} — с учётом регистра")
print("Правила заданы верно." if not msgs else "Исправьте:\n- " + "\n- ".join(msgs))

## Шаг 4. Показатели качества raw-слоя

Выполните ячейку: функции считают C, V, U, S и DQ по определениям из теоретического минимума.

In [ ]:
WEIGHTS = {"C": 0.35, "V": 0.30, "U": 0.20, "S": 0.15}
DICTIONARY = {"TMA", "CMA", "Exam"}                 # допустимые значения assessment_type


def dq(parts):
    """Интегральная учебная метрика качества DQ (лекция 2, раздел 10)."""
    parts["DQ"] = sum(WEIGHTS[k] * parts[k] for k in WEIGHTS)
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in parts.items()}


def quality_raw(df):
    """Показатели качества сырого среза (все поля — строки)."""
    n = len(df)
    missing = df["score"].isna() | df["score"].isin(MISSING_CODES)
    num = pd.to_numeric(df["score"], errors="coerce")
    valid = ~missing & num.between(0, 100)
    withdrawn_exam = (df["final_result"] == "Withdrawn") & \
                     (df["assessment_type"].str.strip().str.upper() == "EXAM")
    return dq({
        "N": n,
        "C": 1 - missing.sum() / n,                          # полнота score
        "V": valid.sum() / (~missing).sum(),                  # валидность среди заполненных
        "U": df[KEY].drop_duplicates().shape[0] / n,          # уникальность ключа
        "S": (df["assessment_type"].isin(DICTIONARY) & ~withdrawn_exam).mean(),  # согласованность
    })


def quality_clean(df):
    """Те же показатели для очищенного среза."""
    n = len(df)
    return dq({
        "N": n,
        "C": df["score"].notna().mean(),
        "V": df["score"].between(0, 100).sum() / df["score"].notna().sum(),
        "U": df[KEY].drop_duplicates().shape[0] / n,
        "S": (~df["withdrawn_exam"]).mean(),
    })


q_raw = quality_raw(raw)
pd.DataFrame([q_raw], index=["raw"])

## Шаг 5. Конвейер очистки raw → clean с журналом строк

Порядок шагов соответствует лекции 2 (раздел 12): **сначала дубли и типы, потом коды и диапазоны,
затем категории и признаки**. Журнал `lineage_df` — основа паспорта данных.

In [ ]:
lineage = []


def log(step, df, note=""):
    lineage.append({"шаг": step, "строк": len(df), "комментарий": note})


log("0. raw-срез варианта", raw)

# 1. Точные дубли: строка повторяется целиком (повторная выгрузка)
step1 = raw.drop_duplicates()
log("1. удалены точные дубли", step1, "повторная выгрузка тех же строк")

# 2. Типы: числовые поля из текста; '?' в сроке сдачи → NaN
step2 = step1.copy()
for col in ["id_assessment", "id_student", "date_submitted", "is_banked"]:
    step2[col] = pd.to_numeric(step2[col], errors="raise").astype("int64")
step2["weight"] = pd.to_numeric(step2["weight"])
step2["assessment_deadline"] = pd.to_numeric(step2["assessment_deadline"].replace("?", np.nan))
log("2. приведены типы", step2, "'?' в assessment_deadline → NaN")

# 3. Логические дубли: одна и та же пара (задание, студент) с разными event_id
step3 = step2.sort_values("event_id").drop_duplicates(subset=KEY, keep="first")
log("3. удалены логические дубли", step3, "оставлена первая доставка (меньший event_id)")

# 4. Служебные коды и доменное правило 0 ≤ score ≤ 100 — строки НЕ удаляем, ставим флаг
step4 = step3.copy()
step4["score_raw"] = step4["score"]
is_code = step4["score_raw"].isna() | step4["score_raw"].isin(MISSING_CODES)
num = pd.to_numeric(step4["score_raw"].where(~is_code), errors="coerce")
in_range = num.between(0, 100)
step4["score_flag"] = np.select([is_code, ~in_range], ["missing_code", "out_of_range"], default="ok")
step4["score"] = num.where(in_range)
log("4. коды пропуска и диапазон score", step4, "значение → NaN + флаг score_flag")

# 5. Нормализация категорий: пробелы, регистр, единый словарь
step5 = step4.copy()
step5["assessment_type"] = step5["assessment_type"].str.strip().str.upper().map(TYPE_MAP)
log("5. нормализованы категории", step5, "tma / Tma → TMA и т. д.")

# 6. Признаки и флаг согласованности
clean = step5.copy()
clean["days_late"] = clean["date_submitted"] - clean["assessment_deadline"]
clean["withdrawn_exam"] = (clean["final_result"] == "Withdrawn") & (clean["assessment_type"] == "Exam")
clean = clean.reset_index(drop=True)
log("6. признаки days_late, withdrawn_exam", clean, "строки не меняются")

lineage_df = pd.DataFrame(lineage)
lineage_df["изменение"] = lineage_df["строк"].diff().fillna(0).astype(int)
lineage_df

In [ ]:
problems = []
if clean.duplicated(subset=KEY).any():
    problems.append("остались повторы ключа (id_assessment, id_student)")
if not clean["score"].dropna().between(0, 100).all():
    problems.append("в score остались значения вне шкалы 0–100")
if clean["assessment_type"].isna().any() or not set(clean["assessment_type"]) <= DICTIONARY:
    problems.append("assessment_type содержит значения вне словаря {TMA, CMA, Exam} — проверьте TYPE_MAP")
if ((clean["score_raw"] == "?") & (clean["score_flag"] != "missing_code")).any():
    problems.append("'?' не распознан как код пропуска — проверьте MISSING_CODES")
if not (clean["score_flag"].eq("ok") == clean["score"].notna()).all():
    problems.append("флаг score_flag не согласован со значением score")
freq = raw_all["score"].value_counts()
frequent_oor = [v for v in clean.loc[clean["score_flag"] == "out_of_range", "score_raw"].unique()
                if freq.get(v, 0) >= 50]
if frequent_oor:
    problems.append(f"значения {frequent_oor} помечены out_of_range, но очень часто встречаются в файле — "
                    "это служебные коды, добавьте их в MISSING_CODES")
removed = (len(raw) - len(step1)) + (len(step2) - len(step3))
if len(clean) != len(raw) - removed:
    problems.append("число строк не сходится с журналом удалений")

print("Все автопроверки пройдены." if not problems else "НАЙДЕНЫ ПРОБЛЕМЫ:\n- " + "\n- ".join(problems))
print("\nФлаги score после очистки:")
print(clean["score_flag"].value_counts().to_string())
print("\nИсходные значения score_raw у флага missing_code:")
print(clean.loc[clean["score_flag"] == "missing_code", "score_raw"].value_counts().to_string())

> **Вопрос.** Почему строки с кодами пропуска и значениями вне шкалы **не удаляются**, а получают флаг?
> Что потерял бы аналитик, если бы просто удалил их?
>
> **Ваш ответ:** ...

## Шаг 6. Качество до и после очистки

In [ ]:
q_clean = quality_clean(clean)
quality = pd.DataFrame([q_raw, q_clean], index=["raw", "clean"])
quality.loc["Δ"] = quality.loc["clean"] - quality.loc["raw"]
display(quality)

dims = ["C", "V", "U", "S", "DQ"]
x = np.arange(len(dims))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, [q_raw[d] for d in dims], width=0.38, label="raw", color="#E45756")
ax.bar(x + 0.2, [q_clean[d] for d in dims], width=0.38, label="clean", color="#4C78A8")
ax.set_xticks(x, dims)
ax.set_ylim(0.97, 1.001)                     # ось обрезана намеренно — подпись ниже
ax.set_ylabel("Значение показателя (ось от 0,97)")
ax.set_title(f"Качество данных до и после очистки: {MODULE} {PRESENTATION}")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

> **Вопрос.** Какие показатели выросли, а какой мог **снизиться**? Объясните, почему после очистки
> полнота C может стать хуже. Обратите внимание на ось графика: с какого значения она начинается и почему.
>
> **Ваш ответ:** ...

## Шаг 7. Выбросы и своевременность

Признак `days_late` — на сколько дней сдача отличается от срока (меньше 0 — раньше срока).
Правило IQR (лекция 2, раздел 7) применяем к работам TMA без перезачтённых результатов.

In [ ]:
tma = clean[(clean["assessment_type"] == "TMA") & (clean["is_banked"] == 0)]["days_late"].dropna()
q1, q3 = tma.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
is_out = (tma < lower) | (tma > upper)

print(f"TMA без перезачтённых (is_banked = 0): {len(tma):,} сдач")
print(f"Q1 = {q1:g}, Q3 = {q3:g}, IQR = {iqr:g}, границы [{lower:g}; {upper:g}]")
print(f"Кандидатов в выбросы: {is_out.sum():,} ({is_out.mean():.1%})")
print(f"  раньше нижней границы: {(tma < lower).sum():,}; позже верхней: {(tma > upper).sum():,}")
print(f"Медиана {tma.median():g} дн., минимум {tma.min():g}, максимум {tma.max():g}")

print("\nПерезачтённые результаты (is_banked) и даты до начала курса:")
display(clean.groupby("is_banked")["date_submitted"]
        .agg(n_rows="count", min_day="min", max_day="max",
             n_negative_day=lambda s: int((s < 0).sum())))

print("Доля сдач позже срока по типам заданий (days_late > 0):")
display(clean.groupby("assessment_type")["days_late"]
        .agg(n_with_deadline="count", share_late=lambda s: round((s > 0).mean(), 3)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tma.clip(-30, 60), bins=90, color="#4C78A8")
ax.axvline(lower, color="#E45756", linestyle="--", label=f"границы IQR: {lower:g} и {upper:g}")
ax.axvline(upper, color="#E45756", linestyle="--")
ax.set_title(f"Отклонение сдачи TMA от срока, {MODULE} {PRESENTATION} (значения обрезаны до −30…60)")
ax.set_xlabel("days_late = date_submitted − assessment_deadline, дней")
ax.set_ylabel("Число сдач")
ax.legend()
plt.tight_layout()
plt.show()

> **Вопросы.**
> 1. Какая доля сдач помечена как кандидаты в выбросы? Можно ли их удалить? Почему?
> 2. Какие даты у перезачтённых результатов (`is_banked = 1`)? Почему их исключают из анализа своевременности?
> 3. Сравните долю «поздних» сдач для TMA и CMA. Правдоподобно ли, что почти все студенты опаздывают с CMA?
>    Что это говорит о смысле поля `assessment_deadline` для тестов?
> 4. Если в вашем варианте есть экзамены (`Exam`): известен ли их срок? Если `days_late` у экзаменов пустой — почему?
>
> **Ваши ответы:** ...

## Шаг 8. Структура пропусков: MCAR или нет?

Лекция 2, раздел 5: если пропуск связан с итогом обучения, удаление таких строк исказит выборку.

In [ ]:
def missing_table(df):
    t = (df.assign(q=df["score_raw"].eq("?"), codes=df["score_raw"].isin(["999", "-1"]))
           .groupby("final_result")
           .agg(n_rows=("event_id", "size"), n_question_mark=("q", "sum"), n_other_codes=("codes", "sum")))
    t["pct_question_mark"] = (100 * t["n_question_mark"] / t["n_rows"]).round(3)
    t["pct_other_codes"] = (100 * t["n_other_codes"] / t["n_rows"]).round(3)
    return t.sort_values("pct_question_mark", ascending=False)


# Весь файл (все 22 презентации): удаляем дубли так же, как в конвейере
all_dedup = (raw_all.drop_duplicates()
             .sort_values("event_id")
             .drop_duplicates(subset=KEY, keep="first")
             .rename(columns={"score": "score_raw"}))
print("Весь файл — доли кодов пропуска по итоговому результату, %:")
display(missing_table(all_dedup))

print(f"Ваш срез {MODULE} {PRESENTATION}:")
display(missing_table(clean))

> **Вопросы.**
> 1. Сравните долю кода `'?'` у групп Withdrawn и Pass во всём файле. Похоже ли это на MCAR?
> 2. Зависит ли доля кодов `999`/`-1` от итога обучения?
> 3. Что произойдёт с выборкой, если удалить все строки с `'?'`?
> 4. Почему выводы по вашему срезу нужно делать осторожно?
>
> **Ваши ответы:** ...

## Шаг 9. BigQuery: raw-слой, профилирование SQL и clean-представление

### Порядок действий в BigQuery Studio

1. Откройте `https://console.cloud.google.com/bigquery` — вверху должна быть метка **Sandbox**.
2. Dataset `learning_analytics` из ПР № 1 можно использовать повторно. Если его нет или он удалён:
   **Explorer → ваш проект → Datasets → Create dataset**; `Dataset ID = learning_analytics`,
   `Location type = Multi-region`, `Location = EU`.
3. `learning_analytics → Create table`:
   - **Source:** `Create table from: Upload`, файл `practice02_raw.csv`, `File format: CSV`;
   - **Destination:** `Table: p02_raw`, `Table type: Native table`;
   - **Schema:** **выключите Auto detect**, включите **Edit as text** и вставьте схему ниже;
   - **Advanced options → Header rows to skip = 1** (щёлкните по `0`, удалите, введите `1`, `Tab`).
4. Откройте `p02_raw → Schema`: все 12 полей должны иметь тип `STRING`; во вкладке **Preview**
   первая строка — данные, а не заголовки.

```json
[
  {"name": "event_id",            "type": "STRING"},
  {"name": "code_module",         "type": "STRING"},
  {"name": "code_presentation",   "type": "STRING"},
  {"name": "id_assessment",       "type": "STRING"},
  {"name": "assessment_type",     "type": "STRING"},
  {"name": "assessment_deadline", "type": "STRING"},
  {"name": "weight",              "type": "STRING"},
  {"name": "id_student",          "type": "STRING"},
  {"name": "date_submitted",      "type": "STRING"},
  {"name": "is_banked",           "type": "STRING"},
  {"name": "score",               "type": "STRING"},
  {"name": "final_result",        "type": "STRING"}
]
```

> **Почему не Auto detect?** Raw-слой хранит выгрузку «как есть». Автоопределение смотрит только
> на начало файла: оно может решить, что `score` — `INTEGER`, и загрузка упадёт на значении `'?'`
> (или потребует разрешить ошибки и молча потеряет строки). Явная схема — это и есть
> **контракт данных** из лекции 2: типы приводятся позже, осознанно, в clean-слое через `SAFE_CAST`.

> **Sandbox:** `CREATE VIEW` и `SELECT` работают; `INSERT/UPDATE/DELETE` — нет, и они не нужны:
> clean-слой строится как **представление (view)** поверх неизменного raw-слоя.


### 9.1. Ваши SQL-запросы

Укажите **Project ID** (TODO-3) и выполните ячейку. Затем `show_sql(...)` напечатает запросы
с вашим проектом и вашим вариантом — копируйте их в редактор BigQuery по одному.

> В исходном тексте запросов стоит пример `BBB 2013J` — функция `sql()` автоматически заменяет его на ваш вариант.

In [ ]:
# TODO-3. Ваш Project ID
PROJECT_ID = "YOUR_PROJECT_ID"   # ← Project ID из BigQuery (не отображаемое имя!)

SQL = {
    'q0_control': '''SELECT
  COUNT(*)                                             AS rows_count,
  COUNT(DISTINCT event_id)                             AS events,
  COUNT(DISTINCT id_student)                           AS students,
  COUNT(DISTINCT CONCAT(code_module, '-', code_presentation)) AS presentations
FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`;''',
    'q1_profile': '''SELECT
  COUNT(*)                                                            AS n_rows,
  COUNTIF(score IS NULL OR score IN ('?', '999', '-1'))               AS n_missing_code,
  COUNTIF(score = '?')                                                AS n_question_mark,
  COUNTIF(SAFE_CAST(score AS INT64) NOT BETWEEN 0 AND 100
          AND score NOT IN ('999', '-1'))                             AS n_out_of_range,
  COUNTIF(assessment_type NOT IN ('TMA', 'CMA', 'Exam'))              AS n_bad_type,
  COUNT(DISTINCT CONCAT(id_assessment, '|', id_student))              AS n_unique_keys
FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`
WHERE code_module = 'BBB' AND code_presentation = '2013J';''',
    'q2_duplicates': '''WITH v AS (
  SELECT * FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`
  WHERE code_module = 'BBB' AND code_presentation = '2013J'
),
no_exact AS (
  SELECT DISTINCT * FROM v
)
SELECT
  (SELECT COUNT(*) FROM v) - (SELECT COUNT(*) FROM no_exact)               AS exact_dups,
  (SELECT COUNT(*) FROM no_exact)
    - (SELECT COUNT(DISTINCT CONCAT(id_assessment, '|', id_student)) FROM no_exact) AS logical_dups;''',
    'q2b_dup_examples': '''WITH no_exact AS (
  SELECT DISTINCT * FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`
  WHERE code_module = 'BBB' AND code_presentation = '2013J'
)
SELECT
  id_assessment,
  id_student,
  COUNT(*)                                     AS n_copies,
  STRING_AGG(event_id, ', ' ORDER BY event_id) AS event_ids,
  STRING_AGG(score, ', ' ORDER BY event_id)    AS scores
FROM no_exact
GROUP BY id_assessment, id_student
HAVING COUNT(*) > 1
ORDER BY id_assessment, id_student
LIMIT 5;''',
    'q3_raw_quality': '''WITH m AS (
  SELECT
    COUNT(*) AS n,
    COUNTIF(score IS NULL OR score IN ('?', '999', '-1'))           AS n_missing,
    COUNTIF(SAFE_CAST(score AS INT64) BETWEEN 0 AND 100)            AS n_valid,
    COUNT(DISTINCT CONCAT(id_assessment, '|', id_student))          AS n_keys,
    COUNTIF(assessment_type IN ('TMA', 'CMA', 'Exam')
            AND NOT (final_result = 'Withdrawn'
                     AND UPPER(TRIM(assessment_type)) = 'EXAM'))    AS n_consistent
  FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`
  WHERE code_module = 'BBB' AND code_presentation = '2013J'
)
SELECT
  n,
  ROUND(1 - n_missing / n, 4)                 AS C,
  ROUND(n_valid / (n - n_missing), 4)         AS V,
  ROUND(n_keys / n, 4)                        AS U,
  ROUND(n_consistent / n, 4)                  AS S,
  ROUND(0.35 * (1 - n_missing / n) + 0.30 * n_valid / (n - n_missing)
      + 0.20 * n_keys / n + 0.15 * n_consistent / n, 4) AS DQ
FROM m;''',
    'q4_create_clean': '''CREATE OR REPLACE VIEW `YOUR_PROJECT_ID.learning_analytics.p02_clean` AS
WITH no_exact AS (                       -- шаг 1: точные дубли
  SELECT DISTINCT * FROM `YOUR_PROJECT_ID.learning_analytics.p02_raw`
),
typed AS (                               -- шаг 2: типы и служебные коды
  SELECT
    event_id,
    code_module,
    code_presentation,
    SAFE_CAST(id_assessment AS INT64)                    AS id_assessment,
    assessment_type                                      AS assessment_type_raw,
    SAFE_CAST(NULLIF(assessment_deadline, '?') AS INT64) AS assessment_deadline,
    SAFE_CAST(weight AS FLOAT64)                         AS weight,
    SAFE_CAST(id_student AS INT64)                       AS id_student,
    SAFE_CAST(date_submitted AS INT64)                   AS date_submitted,
    SAFE_CAST(is_banked AS INT64)                        AS is_banked,
    score                                                AS score_raw,
    IF(score IN ('?', '999', '-1'), NULL, SAFE_CAST(score AS INT64)) AS score_int,
    final_result
  FROM no_exact
),
dedup AS (                               -- шаг 3: логические дубли по ключу
  SELECT * FROM typed
  WHERE TRUE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_assessment, id_student
    ORDER BY event_id
  ) = 1
)
SELECT                                   -- шаги 4–6: правила, нормализация, признаки
  event_id,
  code_module,
  code_presentation,
  id_assessment,
  CASE UPPER(TRIM(assessment_type_raw))
    WHEN 'TMA'  THEN 'TMA'
    WHEN 'CMA'  THEN 'CMA'
    WHEN 'EXAM' THEN 'Exam'
  END                                                    AS assessment_type,
  assessment_deadline,
  weight,
  id_student,
  date_submitted,
  is_banked,
  score_raw,
  IF(score_int BETWEEN 0 AND 100, score_int, NULL)       AS score,
  CASE
    WHEN score_raw IS NULL OR score_raw IN ('?', '999', '-1') THEN 'missing_code'
    WHEN score_int IS NULL OR score_int NOT BETWEEN 0 AND 100 THEN 'out_of_range'
    ELSE 'ok'
  END                                                    AS score_flag,
  final_result,
  date_submitted - assessment_deadline                   AS days_late,
  (final_result = 'Withdrawn'
   AND UPPER(TRIM(assessment_type_raw)) = 'EXAM')        AS withdrawn_exam
FROM dedup;''',
    'q5_clean_quality': '''WITH m AS (
  SELECT
    COUNT(*)                                     AS n,
    COUNTIF(score IS NOT NULL)                   AS n_present,
    COUNTIF(score BETWEEN 0 AND 100)             AS n_valid,
    COUNT(DISTINCT CONCAT(CAST(id_assessment AS STRING), '|',
                          CAST(id_student AS STRING)))  AS n_keys,
    COUNTIF(NOT withdrawn_exam)                  AS n_consistent,
    COUNTIF(score_flag = 'missing_code')         AS n_missing_code,
    COUNTIF(score_flag = 'out_of_range')         AS n_out_of_range
  FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
  WHERE code_module = 'BBB' AND code_presentation = '2013J'
)
SELECT
  n, n_missing_code, n_out_of_range,
  ROUND(n_present / n, 4)      AS C,
  ROUND(n_valid / n_present, 4) AS V,
  ROUND(n_keys / n, 4)         AS U,
  ROUND(n_consistent / n, 4)   AS S,
  ROUND(0.35 * n_present / n + 0.30 * n_valid / n_present
      + 0.20 * n_keys / n + 0.15 * n_consistent / n, 4) AS DQ
FROM m;''',
    'q6_iqr': '''WITH x AS (
  SELECT days_late
  FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
  WHERE code_module = 'BBB' AND code_presentation = '2013J'
    AND assessment_type = 'TMA'
    AND is_banked = 0
    AND days_late IS NOT NULL
),
q AS (
  SELECT DISTINCT
    PERCENTILE_CONT(days_late, 0.25) OVER () AS q1,
    PERCENTILE_CONT(days_late, 0.75) OVER () AS q3
  FROM x
)
SELECT
  q1,
  q3,
  q3 - q1                   AS iqr,
  q1 - 1.5 * (q3 - q1)      AS lower_bound,
  q3 + 1.5 * (q3 - q1)      AS upper_bound,
  COUNTIF(x.days_late < q1 - 1.5 * (q3 - q1)
       OR x.days_late > q3 + 1.5 * (q3 - q1)) AS n_outliers,
  COUNT(*)                  AS n_total
FROM x CROSS JOIN q
GROUP BY q1, q3;''',
    'q6b_banked': '''SELECT
  is_banked,
  COUNT(*)                   AS n_rows,
  MIN(date_submitted)        AS min_day,
  MAX(date_submitted)        AS max_day,
  COUNTIF(date_submitted < 0) AS n_negative_day
FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
WHERE code_module = 'BBB' AND code_presentation = '2013J'
GROUP BY is_banked
ORDER BY is_banked;''',
    'q7_missing_structure': '''SELECT
  final_result,
  COUNT(*)                                         AS n_rows,
  COUNTIF(score_raw = '?')                         AS n_question_mark,
  COUNTIF(score_raw IN ('999', '-1'))              AS n_other_codes,
  ROUND(100 * COUNTIF(score_raw = '?') / COUNT(*), 3) AS pct_question_mark,
  ROUND(100 * COUNTIF(score_raw IN ('999', '-1')) / COUNT(*), 3) AS pct_other_codes
FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
GROUP BY final_result
ORDER BY pct_question_mark DESC;''',
    'q8_check': '''SELECT
  id_assessment,
  assessment_type,
  COUNT(*)                 AS n_rows,
  COUNT(score)             AS n_scored,
  ROUND(AVG(score), 2)     AS mean_score,
  COUNTIF(days_late > 0)   AS n_late
FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
WHERE code_module = 'BBB' AND code_presentation = '2013J'
GROUP BY id_assessment, assessment_type
ORDER BY id_assessment;''',
    'q9_create_analytic': '''CREATE OR REPLACE VIEW `YOUR_PROJECT_ID.learning_analytics.p02_analytic` AS
SELECT
  code_module,
  code_presentation,
  id_student,
  ANY_VALUE(final_result)                     AS final_result,
  COUNT(*)                                    AS n_submissions,
  COUNT(score)                                AS n_scored,
  ROUND(AVG(score), 2)                        AS mean_score,
  ROUND(SAFE_DIVIDE(
    COUNTIF(assessment_type = 'TMA' AND days_late > 0),
    COUNTIF(assessment_type = 'TMA' AND days_late IS NOT NULL)), 3) AS share_late_tma,
  COUNTIF(score_flag != 'ok')                 AS n_flagged
FROM `YOUR_PROJECT_ID.learning_analytics.p02_clean`
GROUP BY code_module, code_presentation, id_student;''',
    'q9b_analytic_check': '''SELECT
  COUNT(*)                          AS students,
  ROUND(AVG(mean_score), 2)         AS avg_mean_score,
  ROUND(AVG(share_late_tma), 3)     AS avg_share_late_tma,
  COUNTIF(n_flagged > 0)            AS students_with_flags
FROM `YOUR_PROJECT_ID.learning_analytics.p02_analytic`
WHERE code_module = 'BBB' AND code_presentation = '2013J';''',
}


VARIANT_FILTER = "code_module = 'BBB' AND code_presentation = '2013J'"


def sql(name):
    """Возвращает запрос с вашим Project ID и фильтром вашего варианта."""
    return (SQL[name]
            .replace("YOUR_PROJECT_ID", PROJECT_ID)
            .replace(VARIANT_FILTER,
                     f"code_module = '{MODULE}' AND code_presentation = '{PRESENTATION}'"))


def show_sql(*names):
    for name in names:
        print(f"-- ===== {name} =====")
        print(sql(name), end="\n\n")

if PROJECT_ID == "YOUR_PROJECT_ID":
    print("Внимание: укажите свой PROJECT_ID, иначе BigQuery ответит Not found.")
else:
    print("Запросы готовы для проекта", PROJECT_ID)

In [ ]:
# Порядок выполнения в BigQuery
show_sql("q0_control", "q1_profile", "q2_duplicates", "q2b_dup_examples", "q3_raw_quality")

In [ ]:
# Создание clean-слоя (выполняется один раз) и проверки после очистки
show_sql("q4_create_clean", "q5_clean_quality", "q6_iqr", "q6b_banked", "q7_missing_structure", "q8_check")

### 9.2. Результаты BigQuery

Перенесите значения из BigQuery и сравните с шагами 2–7 в Colab.

| Показатель | Colab (pandas) | BigQuery | Совпало? |
|---|---|---|---|
| `q0`: строк / событий / презентаций | 176172 / 174781 / 22 |  |  |
| `q2`: точных / логических дублей |  |  |  |
| `q3`: DQ raw |  |  |  |
| `q5`: строк clean / DQ clean |  |  |  |
| `q6`: Q1 / Q3 / число кандидатов |  |  |  |

### 9.3. Собственный SQL-запрос

Запрос `q7_missing_structure` считает доли кодов по **всему** файлу. Измените его так, чтобы он считал
то же самое **только для вашего варианта**, выполните в BigQuery и сравните со второй таблицей шага 8.

```sql
-- ВАШ SQL
```

**Совпал ли результат с Colab?** ...

## Шаг 10. Сверка Python и BigQuery

1. Выполните в BigQuery запрос `q8_check`.
2. **Save results → CSV (local file)**, переименуйте файл в `bq_p02_check.csv`.
3. Выполните ячейки ниже и загрузите файл, когда Colab попросит.

In [ ]:
chk_py = (clean.groupby(["id_assessment", "assessment_type"])
          .agg(n_rows=("event_id", "size"),
               n_scored=("score", "count"),
               mean_score=("score", "mean"),
               n_late=("days_late", lambda s: int((s > 0).sum())))
          .reset_index())
chk_py["mean_score"] = chk_py["mean_score"].round(2)


def compare_with(other, label):
    cols = ["id_assessment", "assessment_type", "n_rows", "n_scored", "mean_score", "n_late"]
    a = chk_py[cols].sort_values("id_assessment").reset_index(drop=True)
    b = other[cols].sort_values("id_assessment").reset_index(drop=True)
    b["id_assessment"] = b["id_assessment"].astype("int64")
    try:
        pd.testing.assert_frame_equal(a, b, check_dtype=False, check_exact=False, atol=0.011, rtol=0)
        print(f"Сверка с {label} пройдена: {len(a)} заданий, все показатели совпали.")
    except AssertionError as e:
        print(f"Расхождение с {label}:\n", str(e)[:800])


chk_py

In [ ]:
# Путь Б (необязательно): выполнить сверочный запрос прямо из Colab через API.
# Работает и в Sandbox. По умолчанию выключено — основной путь: консоль + Save results → CSV.
USE_BQ_API = False

if USE_BQ_API:
    from google.colab import auth
    from google.cloud import bigquery

    auth.authenticate_user()
    client = bigquery.Client(project=PROJECT_ID)
    client.query(sql("q8_check")).to_dataframe().to_csv("bq_p02_check.csv", index=False)
    print("Результат q8_check сохранён в bq_p02_check.csv")
else:
    print("Путь Б выключен: используйте файл, сохранённый из консоли BigQuery.")

In [ ]:
BQ_FILE = Path("bq_p02_check.csv")
if not BQ_FILE.exists():
    try:
        from google.colab import files
        print("Загрузите bq_p02_check.csv — результат запроса q8_check из BigQuery.")
        up = files.upload()
        if up:
            BQ_FILE = Path(next(iter(up)))
    except ImportError:
        pass

if BQ_FILE.exists():
    compare_with(pd.read_csv(BQ_FILE), "BigQuery")
else:
    print("Файл bq_p02_check.csv не найден: выполните q8_check в BigQuery и сохраните результат в CSV.")

## Шаг 11. ★ Аналитический слой (по желанию, не оценивается)

Представление `p02_analytic` — одна строка на студента. Оно понадобится в ЛР № 2.

In [ ]:
show_sql("q9_create_analytic", "q9b_analytic_check")

> Подумайте: можно ли использовать `mean_score` из этого представления, чтобы **в начале курса**
> прогнозировать итог? Что такое утечка данных (лекция 2, раздел 13)?

## Шаг 12. Паспорт набора данных и вывод

### Паспорт (заполните фактическими числами своего варианта)

```text
Набор:           learning_analytics.p02_clean, срез ____ ____
Источник:        OULAD (UCI, CC BY 4.0) + учебная выгрузка practice02_raw.csv
Raw-слой:        learning_analytics.p02_raw, все поля STRING
Строк:           raw ____ → −____ точных дублей → ____ → −____ логических дублей → clean ____
Ключ:            (id_assessment, id_student); при логических дублях оставлена ____________
score:           коды пропуска {________} → NULL (missing_code: ____); вне шкалы → NULL (out_of_range: ____)
Категории:       ____ нестандартных написаний приведены к {TMA, CMA, Exam}
Качество:        DQ ____ → ____; C ____ → ____
Согласованность: записей Withdrawn + Exam — ____
Ограничения:     ______________________________________________
Сверка:          q8_check ↔ pandas: совпало / не совпало
```

### Педагогический вывод (5–7 предложений)

Обязательно ответьте:

1. Насколько исходная выгрузка была пригодна для анализа «как есть»?
2. Как очистка изменила средний балл и число сдач? Почему это важно для выводов об успешности курса?
3. Почему строки с пропусками нельзя просто удалить?
4. Какие ограничения данных нужно сообщить педагогу или администрации?

**Запрещённые формулировки:** «отчисленные не сдают работы», «модель доказала», «удалили плохие данные».

> **Ваш вывод:**
> ...

---
## Чек-лист перед сдачей

- [ ] Заполнены ФИО, группа, номер варианта; `VARIANT` указан.
- [ ] Контрольные значения файла: 176 172 строки, 174 781 `event_id`, 22 презентации.
- [ ] Выполнено профилирование, записаны выводы.
- [ ] `MISSING_CODES` и `TYPE_MAP` заданы — «Правила заданы верно».
- [ ] Конвейер выполнен — «Все автопроверки пройдены».
- [ ] Таблица качества «до / после» и ответ, почему изменилась C.
- [ ] Ответы на вопросы шагов 7 и 8.
- [ ] В BigQuery создана таблица `p02_raw` с явной схемой (все поля `STRING`).
- [ ] Выполнены `q0`–`q8`, создано представление `p02_clean`, заполнена таблица 9.2.
- [ ] Написан собственный SQL-запрос 9.3.
- [ ] Сверка `q8_check` с pandas пройдена.
- [ ] Заполнен паспорт данных, написан вывод без запрещённых формулировок.

### Что сдавать

Сохраните блокнот как `Фамилия_Группа_ПР02.ipynb`: **File / Файл → Download / Скачать → Download .ipynb**.